# Jour 3 — Optuna CatBoost GPU + MLflow

**Objectif** : maximiser le **recall** (ne rater aucun accident grave) tout en
limitant les **faux positifs** (precision acceptable).

Strategie :
- Optimisation multi-critere : **PR AUC** (resume recall + precision) comme metrique principale
- `scale_pos_weight` dans le search space pour gerer le desequilibre 36/64
- Seuil de decision optimise par F-beta (beta=2 → favorise le recall)
- Contrainte : precision >= 0.30 pour limiter les fausses alertes
- GPU accelere (RTX 3060)

## Chargement des donnees

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable")

root = _find_root("out")
path = root / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
assert path.exists(), f"CSV introuvable: {path}"

TARGET = "grave"
SEP = ";"

product15_v2 = [
    "dep", "lum", "atm", "catr", "agg", "int", "circ", "col",
    "vma_bucket", "catv_family_4", "manv_mode", "driver_age_bucket",
    "choc_mode", "driver_trajet_family", "time_bucket",
]
cat_cols = product15_v2[:]
MISSING_CAT = "__MISSING__"

df = pd.read_csv(path, sep=SEP)
X = df[product15_v2].copy()
y = df[TARGET].astype(int).copy()

for c in cat_cols:
    X[c] = X[c].astype("string").fillna(MISSING_CAT).astype(str)

# Ratio de desequilibre
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
natural_weight = n_neg / n_pos
print(f"Dataset : {len(y)} lignes")
print(f"Classe 0 (non grave) : {n_neg} ({100*n_neg/len(y):.1f}%)")
print(f"Classe 1 (grave)     : {n_pos} ({100*n_pos/len(y):.1f}%)")
print(f"Ratio naturel (neg/pos) : {natural_weight:.2f}")

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain : {X_train.shape} | Valid : {X_valid.shape}")

Dataset : 164526 lignes
Classe 0 (non grave) : 105172 (63.9%)
Classe 1 (grave)     : 59354 (36.1%)
Ratio naturel (neg/pos) : 1.77

Train : (131620, 15) | Valid : (32906, 15)


## Configuration MLflow

In [2]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "optuna-catboost-recall"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f"MLflow experiment: {MLFLOW_EXPERIMENT}")

MLflow experiment: optuna-catboost-recall


## Fonctions utilitaires

In [3]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, f1_score, fbeta_score,
    precision_score, recall_score, accuracy_score,
)

def find_best_threshold_fbeta(y_true, proba, beta=2.0):
    """Trouve le seuil qui maximise F-beta (beta>1 favorise recall)."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    # precision_recall_curve retourne un element de plus que thresholds
    precisions = precisions[:-1]
    recalls = recalls[:-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        fbeta = ((1 + beta**2) * precisions * recalls) / (beta**2 * precisions + recalls)
    fbeta = np.nan_to_num(fbeta)
    best_idx = np.argmax(fbeta)
    return float(thresholds[best_idx]), float(fbeta[best_idx])


def compute_metrics(y_true, proba, threshold):
    """Calcule toutes les metriques pour un seuil donne."""
    preds = (proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "recall": recall_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds),
        "f2": fbeta_score(y_true, preds, beta=2),
        "accuracy": accuracy_score(y_true, preds),
    }

print("Fonctions utilitaires chargees")

Fonctions utilitaires chargees


## Optuna — search space GPU-compatible

Params explores :
- `depth` : profondeur des arbres
- `learning_rate` : vitesse d'apprentissage
- `l2_leaf_reg` : regularisation L2
- `random_strength` : bruit d'exploration
- `bagging_temperature` : temperature du bootstrap bayesien
- `border_count` : bins pour les features numeriques
- `min_data_in_leaf` : taille min des feuilles (anti-overfitting)
- `scale_pos_weight` : poids de la classe positive (gestion du desequilibre)
- `grow_policy` : strategie de croissance (SymmetricTree vs Lossguide)

In [4]:
import optuna
import gc
from catboost import CatBoostClassifier

RANDOM_SEED = 42
N_TRIALS = 30
MAX_ITERS = 4000       # reduit (early stopping arrete avant de toute facon)
EARLY_STOP = 200
BETA = 2.0
MIN_PRECISION = 0.30

GPU_INCOMPATIBLE = {"rsm"}


def objective(trial: optuna.Trial) -> float:
    # --- Search space GPU-safe (limites VRAM 6 Go) ---
    depth = trial.suggest_int("depth", 4, 8)  # max 8 au lieu de 10 (VRAM)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.15, log=True)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 0.01, 50.0, log=True)
    random_strength = trial.suggest_float("random_strength", 0.0, 10.0)
    min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 1, 100)
    border_count = trial.suggest_int("border_count", 64, 255)
    scale_pos_weight = trial.suggest_float("scale_pos_weight", 1.0, 3.0)

    # Bootstrap : Bayesian ou MVS
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ["Bayesian", "MVS"])
    if bootstrap_type == "Bayesian":
        bagging_temperature = trial.suggest_float("bagging_temperature", 0.0, 2.0)
        subsample = None
    else:
        bagging_temperature = None
        subsample = trial.suggest_float("subsample", 0.6, 1.0)

    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": MAX_ITERS,
        "random_seed": RANDOM_SEED,
        "verbose": 0,
        "task_type": "GPU",
        "od_type": "Iter",
        "od_wait": EARLY_STOP,
        "depth": depth,
        "learning_rate": learning_rate,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "min_data_in_leaf": min_data_in_leaf,
        "border_count": border_count,
        "scale_pos_weight": scale_pos_weight,
        "bootstrap_type": bootstrap_type,
        "grow_policy": "SymmetricTree",  # Lossguide retire (crash VRAM)
    }
    if bagging_temperature is not None:
        params["bagging_temperature"] = bagging_temperature
    if subsample is not None:
        params["subsample"] = subsample

    model = CatBoostClassifier(**params)
    model.fit(
        X_train, y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    proba = model.predict_proba(X_valid)[:, 1]

    # Seuil optimal F-beta
    best_thr, _ = find_best_threshold_fbeta(y_valid, proba, beta=BETA)
    metrics = compute_metrics(y_valid, proba, best_thr)

    trial.set_user_attr("threshold", best_thr)
    trial.set_user_attr("recall", metrics["recall"])
    trial.set_user_attr("precision", metrics["precision"])
    trial.set_user_attr("f1", metrics["f1"])
    trial.set_user_attr("f2", metrics["f2"])
    trial.set_user_attr("roc_auc", metrics["roc_auc"])
    trial.set_user_attr("best_iteration", model.get_best_iteration())

    # Liberer la VRAM entre chaque trial
    del model
    gc.collect()

    if metrics["precision"] < MIN_PRECISION:
        return 0.0

    return metrics["pr_auc"]


print(f"Search space defini : {N_TRIALS} trials, GPU (VRAM-safe)")
print(f"  depth: 4-8 | iterations: {MAX_ITERS} | grow_policy: SymmetricTree")
print(f"  beta={BETA} | contrainte precision >= {MIN_PRECISION}")

Search space defini : 30 trials, GPU (VRAM-safe)
  depth: 4-8 | iterations: 4000 | grow_policy: SymmetricTree
  beta=2.0 | contrainte precision >= 0.3


## Lancement Optuna

In [5]:
study = optuna.create_study(
    direction="maximize",
    study_name="catboost-recall-pr-auc",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nMeilleur trial #{study.best_trial.number}")
print(f"  PR AUC    : {study.best_value:.4f}")
print(f"  Recall    : {study.best_trial.user_attrs['recall']:.4f}")
print(f"  Precision : {study.best_trial.user_attrs['precision']:.4f}")
print(f"  F2        : {study.best_trial.user_attrs['f2']:.4f}")
print(f"  Threshold : {study.best_trial.user_attrs['threshold']:.3f}")
print(f"  Params    : {study.best_params}")

[I 2026-02-25 13:07:56,301] A new study created in memory with name: catboost-recall-pr-auc


  0%|          | 0/30 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-02-25 13:08:23,125] Trial 0 finished with value: 0.7121797730352477 and parameters: {'depth': 5, 'learning_rate': 0.13125830316209655, 'l2_leaf_reg': 5.100627805979912, 'random_strength': 5.986584841970366, 'min_data_in_leaf': 16, 'border_count': 93, 'scale_pos_weight': 1.116167224336399, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1.416145155592091}. Best is trial 0 with value: 0.7121797730352477.


Default metric period is 5 because AUC is/are not implemented for GPU


...

[I 2026-02-25 13:35:58,342] Trial 27 finished with value: 0.7137191589233897 and parameters: {'depth': 7, 'learning_rate': 0.011587522747984538, 'l2_leaf_reg': 0.06220823273066528, 'random_strength': 6.944295866368336, 'min_data_in_leaf': 90, 'border_count': 172, 'scale_pos_weight': 2.326615328992768, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.29951186039854133}. Best is trial 24 with value: 0.7145910773338269.


Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-02-25 13:37:10,347] Trial 28 finished with value: 0.7137727079896995 and parameters: {'depth': 6, 'learning_rate': 0.031040681585914196, 'l2_leaf_reg': 0.2062636471626572, 'random_strength': 5.286161415509497, 'min_data_in_leaf': 40, 'border_count': 236, 'scale_pos_weight': 2.5848223307700766, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.34590307909563905}. Best is trial 24 with value: 0.7145910773338269.


Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-02-25 13:38:13,928] Trial 29 finished with value: 0.7144259829742649 and parameters: {'depth': 5, 'learning_rate': 0.04642452650349349, 'l2_leaf_reg': 4.241164166907791, 'random_strength': 6.196794741755697, 'min_data_in_leaf': 68, 'border_count': 66, 'scale_pos_weight': 2.109569085636627, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.016063093146534813}. Best is trial 24 with value: 0.7145910773338269.

Meilleur trial #24
  PR AUC    : 0.7146
  Recall    : 0.9221
  Precision : 0.4934
  F2        : 0.7856
  Threshold : 0.304
  Params    : {'depth': 6, 'learning_rate': 0.027504715273577205, 'l2_leaf_reg': 2.8101381544610207, 'random_strength': 7.002382971510138, 'min_data_in_leaf': 66, 'border_count': 144, 'scale_pos_weight': 2.3500437628674695, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.3199484315837708}


## Tableau recapitulatif des trials

In [6]:
trials_df = study.trials_dataframe()
# Colonnes utiles
show_cols = [c for c in trials_df.columns if any(
    k in c for k in ["number", "value", "recall", "precision", "f1", "f2", "roc_auc", "threshold"]
)]
trials_df_show = trials_df[show_cols].sort_values("value", ascending=False).head(15)
display(trials_df_show)

,number,value,user_attrs_f1,user_attrs_f2,user_attrs_precision,user_attrs_recall,user_attrs_roc_auc,user_attrs_threshold
24,24,0.714591,0.642842,0.785583,0.493419,0.922079,0.821448,0.303728
29,29,0.714426,0.648731,0.786001,0.502475,0.915087,0.822111,0.296708
21,21,0.714392,0.647073,0.785509,0.500161,0.916182,0.821604,0.310790
18,18,0.714080,0.647045,0.786124,0.499702,0.917614,0.821164,0.349381
25,25,0.714067,0.641173,0.785608,0.490787,0.924438,0.821580,0.307525
13,13,0.713961,0.649579,0.785414,0.504235,0.912644,0.821313,0.285846
14,14,0.713810,0.645074,0.785734,0.496836,0.919383,0.821525,0.275693
28,28,0.713773,0.647116,0.786055,0.499862,0.917362,0.821513,0.332621
27,27,0.713719,0.651404,0.785653,0.507011,0.910791,0.821444,0.325042
11,11,0.713662,0.635628,0.784321,0.483011,0.929239,0.821136,0.242279


## Re-entrainement du meilleur modele + logging MLflow complet

In [7]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay,
)
import tempfile, os

# Reconstruire les params du meilleur trial (GPU-safe)
best = study.best_trial
best_params = {k: v for k, v in best.params.items()}

retrain_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": MAX_ITERS,
    "random_seed": RANDOM_SEED,
    "verbose": 200,
    "task_type": "GPU",
    "od_type": "Iter",
    "od_wait": EARLY_STOP,
}
retrain_params.update(best_params)

best_model = CatBoostClassifier(**retrain_params)
best_model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
)

proba_best = best_model.predict_proba(X_valid)[:, 1]
best_thr = best.user_attrs["threshold"]
metrics_best = compute_metrics(y_valid, proba_best, best_thr)

print(f"\nModele re-entraine")
for k, v in metrics_best.items():
    print(f"  {k:12s}: {v:.4f}")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7778554	best: 0.7778554 (0)	total: 23.1ms	remaining: 1m 32s
200:	test: 0.8127709	best: 0.8127709 (200)	total: 4.45s	remaining: 1m 24s
400:	test: 0.8160182	best: 0.8160182 (400)	total: 8.78s	remaining: 1m 18s
600:	test: 0.8181877	best: 0.8181877 (600)	total: 13.1s	remaining: 1m 14s
800:	test: 0.8193132	best: 0.8193132 (800)	total: 17.4s	remaining: 1m 9s
1000:	test: 0.8199057	best: 0.8199057 (1000)	total: 21.8s	remaining: 1m 5s
1200:	test: 0.8202862	best: 0.8202862 (1200)	total: 26.2s	remaining: 1m
1400:	test: 0.8205787	best: 0.8205814 (1397)	total: 30.5s	remaining: 56.6s
1600:	test: 0.8207994	best: 0.8208049 (1592)	total: 34.9s	remaining: 52.4s
1800:	test: 0.8210370	best: 0.8210371 (1799)	total: 39.4s	remaining: 48.1s
2000:	test: 0.8211757	best: 0.8211800 (1981)	total: 43.8s	remaining: 43.8s
2200:	test: 0.8212945	best: 0.8212963 (2199)	total: 48.2s	remaining: 39.4s
2400:	test: 0.8213615	best: 0.8214004 (2360)	total: 52.6s	remaining: 35.1s
2600:	test: 0.8214901	best: 0.8214901

In [8]:
# --- Log dans MLflow ---
import mlflow.catboost

# Metadonnees dataset (etape 6)
dataset_meta = {
    "dataset_size": len(y),
    "train_size": len(y_train),
    "valid_size": len(y_valid),
    "n_features": len(product15_v2),
    "class_0_count": int((y == 0).sum()),
    "class_1_count": int((y == 1).sum()),
    "class_1_ratio": round(int(y.sum()) / len(y), 4),
    "class_imbalance_ratio": round(int((y == 0).sum()) / int(y.sum()), 4),
}

with mlflow.start_run(run_name="optuna_best_recall_catboost"):
    mlflow.set_tags({
        "model_family": "catboost",
        "stage": "optuna_recall",
        "optuna_trial": best.number,
        "optimization_target": "pr_auc",
        "threshold_method": f"f_beta_{BETA}",
    })

    # Hyperparametres
    for k, v in best_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("task_type", "GPU")
    mlflow.log_param("n_trials", N_TRIALS)
    mlflow.log_param("beta", BETA)
    mlflow.log_param("min_precision_constraint", MIN_PRECISION)

    # Metadonnees dataset (etape 6)
    mlflow.log_params({f"data_{k}": v for k, v in dataset_meta.items()})

    # Metriques
    mlflow.log_metrics(metrics_best)
    mlflow.log_metric("best_iteration", best_model.get_best_iteration())

    # Artefacts
    with tempfile.TemporaryDirectory() as tmpdir:
        preds_best = (proba_best >= best_thr).astype(int)

        # Matrice de confusion
        fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_valid, preds_best, ax=ax_cm,
            display_labels=["Non grave", "Grave"],
        )
        ax_cm.set_title(f"Confusion Matrix (seuil={best_thr:.3f})")
        cm_path = os.path.join(tmpdir, "confusion_matrix.png")
        fig_cm.savefig(cm_path, dpi=100, bbox_inches="tight")
        plt.close(fig_cm)
        mlflow.log_artifact(cm_path)

        # Courbe ROC
        fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
        RocCurveDisplay.from_predictions(y_valid, proba_best, ax=ax_roc)
        ax_roc.set_title("ROC Curve — Best Optuna")
        roc_path = os.path.join(tmpdir, "roc_curve.png")
        fig_roc.savefig(roc_path, dpi=100, bbox_inches="tight")
        plt.close(fig_roc)
        mlflow.log_artifact(roc_path)

        # Courbe Precision-Recall
        fig_pr, ax_pr = plt.subplots(figsize=(6, 5))
        PrecisionRecallDisplay.from_predictions(y_valid, proba_best, ax=ax_pr)
        ax_pr.axhline(y=MIN_PRECISION, color="red", linestyle="--", label=f"min precision={MIN_PRECISION}")
        ax_pr.axvline(x=metrics_best["recall"], color="green", linestyle="--", alpha=0.5, label=f"recall={metrics_best['recall']:.3f}")
        ax_pr.legend()
        ax_pr.set_title("Precision-Recall Curve")
        pr_path = os.path.join(tmpdir, "precision_recall_curve.png")
        fig_pr.savefig(pr_path, dpi=100, bbox_inches="tight")
        plt.close(fig_pr)
        mlflow.log_artifact(pr_path)

        # Feature importance
        fi = best_model.get_feature_importance()
        fi_df = pd.DataFrame({
            "feature": product15_v2,
            "importance": fi,
        }).sort_values("importance", ascending=False)
        fi_path = os.path.join(tmpdir, "feature_importance.csv")
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        # Feature names (etape 6)
        fn_path = os.path.join(tmpdir, "feature_names.txt")
        with open(fn_path, "w") as f:
            f.write("\n".join(product15_v2))
        mlflow.log_artifact(fn_path)

        # Optuna history
        trials_export = study.trials_dataframe()
        trials_path = os.path.join(tmpdir, "optuna_trials.csv")
        trials_export.to_csv(trials_path, index=False)
        mlflow.log_artifact(trials_path)

    # Modele
    mlflow.catboost.log_model(best_model, artifact_path="model")

print("Run MLflow logge : optuna_best_recall_catboost")
print(f"Voir dans MLflow UI > experience '{MLFLOW_EXPERIMENT}'")

2026/02/25 13:39:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run optuna_best_recall_catboost at: http://127.0.0.1:5000/#/experiments/3/runs/f66bc2e7987d4740bcfe6b03f1986662
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Run MLflow logge : optuna_best_recall_catboost
Voir dans MLflow UI > experience 'optuna-catboost-recall'


## Export .cbm + meta.json (pour predictor.py)

In [2]:
import json
from datetime import datetime
from pathlib import Path
import mlflow
import mlflow.catboost

# --- Config autonome (pas besoin d'executer les cellules precedentes) ---
def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable")

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
OPTUNA_RUN_ID = "f66bc2e7987d4740bcfe6b03f1986662"

FEATURES = [
    "dep", "lum", "atm", "catr", "agg", "int", "circ", "col",
    "vma_bucket", "catv_family_4", "manv_mode", "driver_age_bucket",
    "choc_mode", "driver_trajet_family", "time_bucket",
]

# --- Chargement depuis MLflow ---
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.MlflowClient()
run = client.get_run(OPTUNA_RUN_ID)

print("Chargement du modele depuis MLflow...")
model = mlflow.catboost.load_model(f"runs:/{OPTUNA_RUN_ID}/model")

# Recuperer metriques et params du run
metrics = run.data.metrics
params = run.data.params
threshold = metrics.get("threshold", 0.5)

# --- Export .cbm ---
root = _find_root("out")
model_name = "catboost_optuna_best"
cbm_path = root / "model" / f"{model_name}.cbm"
meta_path = root / "artifacts" / f"{model_name}_meta.json"
(root / "model").mkdir(exist_ok=True)
(root / "artifacts").mkdir(exist_ok=True)

model.save_model(str(cbm_path))

# --- Export meta.json ---
catboost_param_keys = [
    "depth", "learning_rate", "l2_leaf_reg", "random_strength",
    "min_data_in_leaf", "border_count", "scale_pos_weight",
    "bootstrap_type", "bagging_temperature", "subsample",
]
catboost_params = {}
for k in catboost_param_keys:
    if k in params:
        val = params[k]
        try:
            val = float(val)
            if val == int(val):
                val = int(val)
        except (ValueError, OverflowError):
            pass
        catboost_params[k] = val

meta = {
    "model_name": model_name,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "threshold": threshold,
    "features": FEATURES,
    "cat_features": FEATURES,
    "catboost_params": catboost_params,
    "metrics": {
        k: round(v, 6) for k, v in metrics.items()
        if k in ("threshold", "pr_auc", "roc_auc", "recall",
                 "precision", "f1", "f2", "accuracy")
    },
    "mlflow_run_id": OPTUNA_RUN_ID,
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"Modele exporte : {cbm_path}")
print(f"Meta exporte   : {meta_path}")
print(f"Threshold      : {threshold:.4f}")
print(f"\nUtilisation dans predictor.py :")
print(f"  MODEL_PATH={cbm_path}")
print(f"  META_PATH={meta_path}")

Chargement du modele depuis MLflow...


Modele exporte : /home/maxime/simplonalternance/alternance-CICDprediction/model/catboost_optuna_best.cbm
Meta exporte   : /home/maxime/simplonalternance/alternance-CICDprediction/artifacts/catboost_optuna_best_meta.json
Threshold      : 0.3037

Utilisation dans predictor.py :
  MODEL_PATH=/home/maxime/simplonalternance/alternance-CICDprediction/model/catboost_optuna_best.cbm
  META_PATH=/home/maxime/simplonalternance/alternance-CICDprediction/artifacts/catboost_optuna_best_meta.json


## Visualisations Optuna

In [9]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
)

plot_optimization_history(study).show()
plot_param_importances(study).show()
plot_slice(study, params=["depth", "learning_rate", "scale_pos_weight", "l2_leaf_reg"]).show()

## Resume final

In [10]:
print("=" * 60)
print("MEILLEUR MODELE OPTUNA — FOCUS RECALL")
print("=" * 60)
print(f"  PR AUC     : {metrics_best['pr_auc']:.4f}")
print(f"  ROC AUC    : {metrics_best['roc_auc']:.4f}")
print(f"  Recall     : {metrics_best['recall']:.4f}  <- objectif principal")
print(f"  Precision  : {metrics_best['precision']:.4f}  <- contrainte >= {MIN_PRECISION}")
print(f"  F1         : {metrics_best['f1']:.4f}")
print(f"  F2         : {metrics_best['f2']:.4f}  <- beta={BETA}")
print(f"  Seuil      : {metrics_best['threshold']:.3f}")
print(f"  Best iter  : {best_model.get_best_iteration()}")
print(f"\nHyperparametres :")
for k, v in sorted(best_params.items()):
    print(f"  {k}: {v}")

MEILLEUR MODELE OPTUNA — FOCUS RECALL
  PR AUC     : 0.7147
  ROC AUC    : 0.8216
  Recall     : 0.9201  <- objectif principal
  Precision  : 0.4948  <- contrainte >= 0.3
  F1         : 0.6435
  F2         : 0.7851  <- beta=2.0
  Seuil      : 0.304
  Best iter  : 3537

Hyperparametres :
  bagging_temperature: 0.3199484315837708
  bootstrap_type: Bayesian
  border_count: 144
  depth: 6
  l2_leaf_reg: 2.8101381544610207
  learning_rate: 0.027504715273577205
  min_data_in_leaf: 66
  random_strength: 7.002382971510138
  scale_pos_weight: 2.3500437628674695
